# 08 · Final models, export and test

Everything here goes through the same trainer as the search, with `[Final]` of the configuration:

- **the winner** of the search trained again, 40,000 batches, seeds 0, 1 and 2, in FP16 with distillation and BatchNorm folded for the last 20% of the steps;
- **baselines**: U-Nets of segmentation_models_pytorch with the EfficientNet-B0 and MobileNetV2 encoders, off the shelf and with the decoder slimmed by hand to fit 5 MB, trained the same way;
- **ablations** of the winner: without distillation (`no_kd`), and trained in FP32 with BatchNorm frozen instead of folded, converted to FP16 only afterwards (`fp32`), as a network is usually brought to the Myriad.

All are scored on the search and validation sets. The deployed model is the winner's seed with the best validation score in FP16; it is exported as the files ArgOS takes and checked. The test split is read once, at the end, by `final.test`, which keeps its result and never computes it again.

In [14]:
# [1] setup
%cd ~/projects/iride_onboard-burnscar-mapper
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner
from pynas.hardware import myriad

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
CONFIG = Path("configs/burnscar.ini")
cfg = config.load(CONFIG)
BASE = Path(cfg.get("Run", "out"))
SEARCH, FINAL, TEACHER_Q = BASE / "search", BASE / "final", BASE / "teacher" / "runs" / "queue.json"
GPUS = list(range(torch.cuda.device_count())) or [0]
w = final.winner(SEARCH)
print(f"PyNAS {pynas.__version__}; winner {w['spec']} (search score {w['score']:.4f}, {w.get('latency_ms')} ms)")

/root/projects/iride_onboard-burnscar-mapper
PyNAS 1.0.1; winner E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1 (search score 0.6849, 40.766 ms)


In [2]:
# [2] the final runs, in the background (running the cell again carries on)
FQ = final.queue(cfg, SEARCH, FINAL)
print(f"{len(runner.read_queue(FQ)['runs'])} runs")
runner.launch(FQ, gpus=GPUS)

/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


18 runs
18 run(s) to do, 2 worker(s): GPU 0 (pid 1413369), GPU 1 (pid 1413371); logs in /root/projects/iride_onboard-burnscar-mapper/models/pynas/final/runs/log_gpu<k>.txt


[{'gpu': 0,
  'pid': 1413369,
  'started': '2026-10-15 00:11:19',
  'loader_workers': 3},
 {'gpu': 1,
  'pid': 1413371,
  'started': '2026-10-15 00:11:19',
  'loader_workers': 3}]

In [9]:
# [3] progress every 15 minutes (interrupting stops the watching, not the trainings)
runner.wait(FQ, every=900)

2026-10-15 07:37:22   18 done


,state,gpu,step,role,label,seed,minutes,score
run,,,,,,,,
r_02c00eaecc75,done,<NA>,40000/40000,final,winner,0,30.106017,0.599262
r_7a5480c13cc4,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0,0,47.671133,0.658571
r_d882c7e0a965,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,0,39.419409,0.626013
r_d6c188270f3c,done,<NA>,40000/40000,baseline,smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8,0,34.206897,0.521891
r_062a27f73fb6,done,<NA>,40000/40000,ablation,"winner, no_kd",0,29.891325,0.665183
r_77a6c1bbda70,done,<NA>,40000/40000,ablation,"winner, fp32",0,28.583044,0.543760
r_ec75a2aa99c5,done,<NA>,40000/40000,final,winner,1,30.957446,0.568405
r_edb41e8018a1,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0,1,47.533446,0.677971
r_fc56d3de2827,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,1,39.007250,0.610528


,state,gpu,step,role,label,seed,model,minutes,score
run,,,,,,,,,
r_02c00eaecc75,done,<NA>,40000/40000,final,winner,0,E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1,30.106017,0.599262
r_7a5480c13cc4,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0,0,smp:Unet:efficientnet-b0,47.671133,0.658571
r_d882c7e0a965,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,0,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,39.419409,0.626013
r_d6c188270f3c,done,<NA>,40000/40000,baseline,smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8,0,smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8,34.206897,0.521891
r_062a27f73fb6,done,<NA>,40000/40000,ablation,"winner, no_kd",0,E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1,29.891325,0.665183
r_77a6c1bbda70,done,<NA>,40000/40000,ablation,"winner, fp32",0,E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1,28.583044,0.543760
r_ec75a2aa99c5,done,<NA>,40000/40000,final,winner,1,E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1,30.957446,0.568405
r_edb41e8018a1,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0,1,smp:Unet:efficientnet-b0,47.533446,0.677971
r_fc56d3de2827,done,<NA>,40000/40000,baseline,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,1,smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8,39.007250,0.610528


In [4]:
# [4] fresh-burn IoU in FP16 on the search and validation sets: mean and standard deviation over the seeds
report.final_table(FQ, cfg).round(3)

seeds heo_search_fp16_labels_fresh_iou        heo_val_fp16_labels_fresh_iou        phisat2_val_fp16_labels_fresh_iou         \
                                                                                      mean    std                          mean    std                              mean    std   
label                                                                                                                                                                             
winner                                                  3                            0.210  0.028                         0.527  0.017                             0.891  0.002   
smp:Unet:efficientnet-b0                                3                            0.415  0.062                         0.561  0.027                             0.887  0.002   
smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8     3                            0.349  0.013                         0.561  0.046                             0.889  0.002   
smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8        3                            0.319  0.018                         0.512  0.098                             0.883  0.003   
winner, no_kd                                           3                            0.347  0.108                         0.483  0.090                             0.885  0.008   
winner, fp32                                            3                            0.337  0.048                         0.478  0.042                             0.888  0.004   

                                                    heo_search_fp16_labels_fresh_iou_scene_mean        heo_val_fp16_labels_fresh_iou_scene_mean        phisat2_val_fp16_labels_fresh_iou_scene_mean         
                                                                                           mean    std                                     mean    std                                         mean    std  
label                                                                                                                                                                                                       
winner                                                                                    0.586  0.016                                    0.527  0.017                                        0.814  0.003  
smp:Unet:efficientnet-b0                                                                  0.672  0.012                                    0.561  0.027                                        0.811  0.005  
smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8                                       0.632  0.025                                    0.561  0.046                                        0.810  0.006  
smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8                                          0.539  0.016                                    0.512  0.098                                        0.799  0.004  
winner, no_kd                                                                             0.572  0.117                                    0.483  0.090                                        0.813  0.007  
winner, fp32                                                                              0.576  0.030                                    0.478  0.042                                        0.811  0.005

## Export

The winner's seed with the best mean validation IoU in FP16, as `models/onboard/burnscar_pynas/`: FP16 IR of the OpenVINO 2022.3.1 Model Optimizer (`.xml`, `.bin`, `.mapping`), checked by compiling it for MYRIAD; its IR run on the OpenVINO CPU plugin against PyTorch on validation windows (a check of the conversion); the share of pixels where FP16 gives the class FP32 gives.

In [5]:
# [5] the deployed model
dep = final.deploy(FQ, cfg, "models/onboard", "burnscar_pynas")
print(json.dumps({k: v for k, v in dep.items() if k not in ("layers", "ir_vs_pytorch")}, indent=1, default=str))
if dep.get("ir_vs_pytorch"):
    display(pd.DataFrame(dep["ir_vs_pytorch"]).set_index("set"))

/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_graph_shape_type_inference(
/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1187: UserWarning: Constant folding

{
 "name": "burnscar_pynas",
 "run": "r_02c00eaecc75",
 "seed": 0,
 "model": "E:eb0:4/L:mb:48:3:4:silu:max/L:res:128:5:1:silu:max/D:conv:16:1",
 "val_fp16": 0.6776226572754467,
 "rule": "winner seed with the best mean validation fresh-burn IoU in FP16 (scene_mean)",
 "time": "2026-10-15 05:41:51",
 "ir_mb": 2.559,
 "compiled": true,
 "mapping": true,
 "files": [
  "burnscar_pynas.bin",
  "burnscar_pynas.blob",
  "burnscar_pynas.mapping",
  "burnscar_pynas.onnx",
  "burnscar_pynas.xml"
 ],
 "argos_unlisted": [],
 "error": null,
 "fp16_agreement": {
  "heo_val": 0.9999782580804747,
  "phisat2_val": 0.9998352755364056
 }
}


,windows,largest_logit,largest_difference,same_class
set,,,,
heo_val,8,14.770059,0.059160,1.000000
phisat2_val,8,16.970846,0.033771,0.999849


In [ ]:
# [6] the hardware numbers of the winner and the baselines (one seed each): parameters of the exported network, IR size, compiled for MYRIAD, latency
from pynas.hardware.latency import latency_ms
from pynas.space.network import n_params_used
res = final.results(FQ)
one = res[res.role.isin(["final", "baseline"])].drop_duplicates("label")
nets = {f"m{i}": runner.load_result(FINAL / "runs" / r.run) for i, r in enumerate(one.itertuples())}
shape = tuple(json.load(open(FINAL / "runs" / one.run.iloc[0] / "run.json"))["input_shape"])
hw = myriad.check(nets, FINAL / "hardware", shape, cfg) if myriad.status(cfg)["model_optimizer"] else {}
lat = latency_ms(nets, shape, cfg) if myriad.status(cfg)["nnmeter"] else {}
pd.DataFrame([dict(model=r.label, params_M=n_params_used(nets[k], shape) / 1e6, ir_mb=hw.get(k, {}).get("ir_mb"),
                   compiled=hw.get(k, {}).get("compiled"), latency_ms=lat.get(k))
              for k, r in zip(nets, one.itertuples())]).set_index("model").round(3)

## Test

Fresh-burn IoU and burn IoU (fresh + old) on every test set: the onboard models in FP16 (as they run on the Myriad) and in FP32, the teacher in FP32. Mean and standard deviation over the seeds; the deployed seed is one of the winner's. Run once; afterwards the cell reads the stored result.

In [15]:
# [7] the test split, read once
scores, per_gsd = final.test(FQ, cfg, teacher_queue=TEACHER_Q)
deployed = scores[scores.run == dep["run"]].assign(label="deployed (" + scores.label + f", seed {dep['seed']})")
display(report.test_table(pd.concat([scores, deployed]), cfg).round(3))

runs heo_test_labels_fresh_iou        phisat2_test_labels_fresh_iou        s2_test_labels_fresh_iou        heo_test_labels_fresh_iou_scene_mean         \
                                                                                        mean    std                          mean    std                     mean    std                                 mean    std   
label                                               precision                                                                                                                                                          
winner                                              fp16         3                     0.516  0.016                         0.883  0.003                    0.748  0.040                                0.531  0.023   
                                                    fp32         3                     0.516  0.016                         0.883  0.003                    0.748  0.040                                0.531  0.023   
smp:Unet:efficientnet-b0                            fp16         3                     0.525  0.019                         0.879  0.002                    0.789  0.039                                0.543  0.041   
                                                    fp32         3                     0.525  0.020                         0.879  0.002                    0.789  0.040                                0.543  0.041   
smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8 fp16         3                     0.520  0.013                         0.878  0.003                    0.740  0.078                                0.533  0.028   
                                                    fp32         3                     0.520  0.013                         0.878  0.003                    0.740  0.078                                0.533  0.028   
smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8    fp16         3                     0.409  0.040                         0.873  0.002                    0.592  0.028                                0.469  0.017   
                                                    fp32         3                     0.409  0.040                         0.873  0.002                    0.592  0.029                                0.469  0.016   
winner, no_kd                                       fp16         3                     0.435  0.050                         0.867  0.006                    0.789  0.040                                0.470  0.043   
                                                    fp32         3                     0.435  0.049                         0.867  0.007                    0.789  0.040                                0.470  0.043   
winner, fp32                                        fp16         3                     0.518  0.015                         0.874  0.003                    0.750  0.024                                0.537  0.010   
                                                    fp32         3                     0.518  0.015                         0.874  0.003                    0.750  0.023                                0.537  0.010   
teacher                                             fp32         2                     0.472  0.007                         0.875  0.001                    0.543  0.065                                0.511  0.002   
teacher, mean of seeds                              fp32         1                     0.485    NaN                         0.880    NaN                    0.565    NaN                                0.529    NaN   
deployed (winner, seed 0)                           fp16         1                     0.501    NaN                         0.880    NaN                    0.715    NaN                                0.505    NaN   
                                                    fp32         1                     0.502    NaN                         0.880    NaN                    0.715    NaN      

In [12]:
report.final_table(FQ, cfg).round(4).to_csv("final_table.csv")
report.test_table(pd.concat([scores, deployed]), cfg).round(4).to_csv("test_table.csv")
per_gsd.to_csv("test_per_gsd.csv", index=False)

In [13]:
# [8] fresh-burn IoU per resolution on the test sets (mean over seeds)
per_gsd.groupby(["label", "precision", "set", "gsd"], sort=False).iou.mean().unstack(["set", "gsd"]).round(3)

set                                                           heo_test                      phisat2_test               s2_test       
gsd                                                                2.5    5.0   10.0    all         4.75   10.0    all    10.0    all
label                                               precision                                                                        
winner                                              fp16         0.516  0.536  0.454  0.516        0.887  0.865  0.883   0.748  0.748
                                                    fp32         0.517  0.536  0.454  0.516        0.887  0.865  0.883   0.748  0.748
smp:Unet:efficientnet-b0                            fp16         0.524  0.537  0.499  0.525        0.882  0.867  0.879   0.789  0.789
                                                    fp32         0.524  0.537  0.499  0.525        0.882  0.867  0.879   0.789  0.789
smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8 fp16         0.511  0.571  0.501  0.520        0.879  0.868  0.878   0.740  0.740
                                                    fp32         0.511  0.571  0.501  0.520        0.879  0.868  0.878   0.740  0.740
smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8    fp16         0.404  0.431  0.410  0.409        0.875  0.859  0.873   0.592  0.592
                                                    fp32         0.404  0.431  0.410  0.409        0.876  0.859  0.873   0.592  0.592
winner, no_kd                                       fp16         0.418  0.529  0.450  0.435        0.870  0.854  0.867   0.789  0.789
                                                    fp32         0.418  0.528  0.450  0.435        0.870  0.854  0.867   0.789  0.789
winner, fp32                                        fp16         0.513  0.556  0.476  0.518        0.877  0.862  0.874   0.750  0.750
                                                    fp32         0.513  0.557  0.476  0.518        0.877  0.862  0.874   0.750  0.750
teacher                                             fp32         0.468  0.499  0.439  0.472        0.881  0.847  0.875   0.543  0.543
teacher, mean of seeds                              fp32         0.482  0.513  0.448  0.485        0.886  0.854  0.880   0.565  0.565

## Test results by resolution and by scene

Fresh-burn IoU on each test set over all its windows and at each resolution it is scored at, mean and standard deviation over the seeds (the onboard models in FP16, the teacher in FP32). HEO scenes are scored at 2.5, 5 and 10 m: the HEO image resampled to windows of that ground size, as the training windows are drawn between 1 and 10 m. PhiSat-2 is scored at 4.75 m, its own resolution, and at 10 m; Sentinel-2 at its own 10 m.

Then the test scenes themselves, every window put back in its place: the image (NIR, red, green), the labels, and for the deployed PyNAS model, the two slimmed U-Nets and the teacher (mean of its two seeds) the fresh burn found (yellow), the false alarms (magenta) and the misses (cyan), with the IoU of that scene at that resolution. For PhiSat-2, the three scenes with the most fresh burn among its sampled test windows; the gaps are windows outside the sample.

In [ ]:
# [9] fresh-burn IoU on every test set, over all windows and per resolution
ORDER = ["winner", "smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8", "smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8",
         "smp:Unet:efficientnet-b0", "winner, no_kd", "winner, fp32", "teacher"]
report.plot_test_iou(per_gsd, labels=ORDER)
plt.tight_layout(); plt.savefig(FINAL / "test_iou_by_gsd.png", dpi=200); plt.show()

In [ ]:
# [10] the test scenes
from pynas import teacher
from pynas.train import data as D
spec = D.DataSpec.from_config(cfg)
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
dep = json.load(open("models/onboard/burnscar_pynas/deploy.json"))
res = final.results(FQ)
seed0 = lambda label: runner.load_result(FINAL / "runs" / res[(res.label == label) & (res.seed == 0)].run.iloc[0], dev)
models = {f"PyNAS (deployed, seed {dep['seed']})": (runner.load_result(FINAL / "runs" / dep["run"], dev), "fp16"),
          "U-Net EfficientNet-B0, slim": (seed0("smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8"), "fp16"),
          "U-Net MobileNetV2, slim": (seed0("smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8"), "fp16"),
          "teacher (mean of seeds)": (final.Mean(teacher.models(TEACHER_Q, dev)), "fp32")}
for name in spec.test:
    s = D.open_set(spec, name, teacher=False)
    fig = report.plot_scenes(s, report.predict_set(models, spec, s, dev), spec, name, max_scenes=3)
    fig.savefig(FINAL / f"test_scenes_{name}.png", dpi=200); plt.show()

Next: the ArgOS test suite on `models/onboard/burnscar_pynas/` (its `.xml`, `.bin` and `.mapping` go to `argos_sdk/models/`), and the model on the Myriad X itself.